# 6장 실습: Python에서 Neo4j 연결하기

실습 가이드 6장 6-1 ~ 6-3의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | Recommendations (4장과 같은 DB. Docker `bolt://localhost:7687` 또는 Recommendations 샌드박스) |
| .env 설정 | Recommendations DB의 NEO4J 네 줄 (실습 가이드 5-5) |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

## 1. 환경 확인 (실습 가이드 6-1)

In [ ]:
from tools.graphrag_tools import check_env

# 커널 경로(.venv), 작업 위치(프로젝트 루트), 패키지 버전을 출력한다.
check_env(["python-dotenv", "neo4j", "langchain-neo4j"])

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [ ]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 Neo4jGraph를 만들고 레이블별 노드 수를 출력한다(실습 가이드 6-2).
# 함수 안에서 하는 일: .env의 NEO4J 네 값을 딕셔너리 NEO4J로 묶고 Neo4jGraph(**NEO4J)로 연결한다.
# expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
graph, NEO4J = connect_graph(expect=["Movie", "User", "Genre"])

## 2. 스키마 확인 (실습 가이드 6-2)

`graph.schema`는 노드 속성, 관계 속성, 관계 패턴을 문자열로 정리한 것입니다. 3장 `CALL db.schema.visualization`이 그림으로 보여 준 구조를 텍스트로 나타낸 것이고, 8장에 LLM에게 넘기는 스키마가 이 문자열입니다.

In [ ]:
print(graph.query("RETURN 'connected' AS status"))
print(graph.schema)

## 3. Browser 쿼리를 Python에서 실행 (실습 가이드 6-3)

Browser에 입력한 쿼리를 `graph.query()`에 문자열로 넣으면 그대로 실행됩니다. 결과는 딕셔너리의 리스트이고, `RETURN`에서 `AS`로 붙인 이름이 키가 됩니다.

Recommendations의 개봉 연도는 `year`(숫자)입니다. `released`는 `'1995-11-22'` 같은 문자열이라 `released >= 2000`으로 비교하면 결과가 0행입니다(실습 가이드 4-1 "Movies와 다른 점").

In [ ]:
rows = graph.query("""
MATCH (m:Movie)
WHERE m.year >= 2000
RETURN m.title AS title, m.year AS year
ORDER BY m.year DESC
LIMIT 5
""")
for row in rows:
    print(row["title"], row["year"])

## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제 아래의 빈 셀에 코드를 작성합니다.

### 과제 1. 3장 쿼리를 Python에서 실행

3장 실습 과제의 쿼리 세 개(Tom Hanks 출연작, 2000년 이후 개봉작 10편, Keanu Reeves 공동 출연자)를 Recommendations DB에 맞게 고쳐 `graph.query()`로 실행하고, 결과를 한 줄에 한 건씩 출력하세요. 개봉 연도는 `released` 대신 `year`를 씁니다. 결과 수가 3장(Movies)과 어떻게 다른지도 확인합니다.

In [ ]:
# 과제 1: 여기에 작성합니다


### 과제 2. 스키마에서 Movie 속성 찾기

`graph.schema` 출력에서 Movie 노드의 속성 목록을 찾으세요. 문자열에서 찾는 방법과, 구조화된 스키마(`graph.structured_schema`)에서 꺼내는 방법을 모두 해 봅니다.

In [ ]:
# 과제 2: 여기에 작성합니다
